In [1]:
import numpy as np
import tensorflow as tf
import keras as keras
import os
import csv

from datetime import datetime
from keras.activations import relu, elu, linear, sigmoid

#from tensorflow import keras
from keras import layers
from keras import models
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, History

from data_toolsFunctional import load_preprocessed, dataPrep, nameModel

#simPrefix = os.getcwd()+'\\simdata'


In [2]:
simPrefix = 'C:\\Users\\mkrit\\Documents\\PHY\\sim_data'
## location for sim files
x, y = load_preprocessed(simPrefix, 'train') 

#nan meants not a number

Percentage of events with a NaN: 2.68


In [3]:
print(x.shape)
##NumPy arrays have an attribute called shape that returns a tuple with each index having the number of corresponding elements.

print(y.keys()) ##don't understand what all the dictionary keys are.

##The keys() method returns a view object. The view object contains the keys of the dictionary, as a list.
##Dictionaries are used to store data values in key:value pairs.

    # each station has 2 tanks, each tank has 2 DOMs (high/log gain)\n",
    # each tank measures charge and time\n",
    # each station gives 2 charges and 2 times, 4 total pieces of data per station\n",
    # stations arranged in 10x10 square lattice, 2 corners of square unused\n",
    # charge measured in VEM, vertical equivalent muon\n",
    # 'dir' is true direction, rest of dir are reconstruted by simulations\n",
    # 'plane_dir' assumes shower is flat plane\n",
    # 'laputop_dir' performs likelihood analysis\n",
    # 'small_dir' compromises between plane and laputop"

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [4]:
# 85/15 split for training/testing
## ask what this is


In [5]:
# Name for model
key = 'Functional'
i = 0
while(os.path.exists('model/model_{}.h5'.format(key+str(i)))):
    i = i + 1
key = key+str(i)
numepochs = 6
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
#why is time 1 at eight thousand
# why are their the two distributions
# if we shift everything hoefully it will be a little more clear.

prep = {'q':None, 't':False, 'normed':True, 'reco':'small', 'cosz':True}

# Establishing arrays to be trained on
x_i = dataPrep(x, y, **prep)

energy = y['energy']
comp = y['comp']

theta, phi = y['dir'].transpose() ##dir shows all attributes of an object
## For an array a with two axes, transpose(a) gives the matrix transpose.

nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)

temp_y = energy

In [6]:
# Create model
## CNN Deep Learning algorithm which can take in an input image, assign importance
##     (learnable weights and biases) to various aspects/objects in the image and be able to differentiate one from the other. 

#conv1_layer = layers.Conv2D(128,kernel_size=3,activation='relu')(charge_input)
#conv2_layer = layers.Conv2D(34,kernel_size=3,activation='relu')(conv1_layer)
#conv3_layer = layers.Conv2D(4,kernel_size=3,activation='relu')(conv2_layer)
#flat_layer = layers.Flatten()(conv3_layer)


charge_input=keras.Input(shape=(10,10,2,),name="charge")

conv1_layer = layers.Conv2D(64,kernel_size=3,activation='relu')(charge_input)
conv2_layer = layers.Conv2D(34,kernel_size=3,activation='relu')(conv1_layer)
conv3_layer = layers.Conv2D(4,kernel_size=3,activation='relu')(conv2_layer)
flat_layer = layers.Flatten()(conv3_layer)

# inputting zenith here
zenith_input=keras.Input(shape=(1,),name="zenith")
print(zenith_input.shape)
#A concatenation layer takes inputs and concatenates them along a specified dimension. 
#The Concat layer is a utility layer that concatenates its multiple input blobs to one single output blob.
concat_layer = layers.concatenate([flat_layer,zenith_input])

#output = layers.Dense(1)(concat_layer)

dense1_layer = layers.Dense(128,activation ='relu')(concat_layer)
dropout_layer1 = layers.Dropout(0.5, input_shape = (2,))(dense1_layer)
dense2_layer = layers.Dense(128,activation ='relu')(dense1_layer)
dropout_layer2 = layers.Dropout(0.5, input_shape = (2,))(dense2_layer)
output = layers.Dense(1)(dense2_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=key)

#talos.Scan(x, y, model = Functional, params = 
#compiling the model
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])
model.summary()


(None, 1)
Model: "Functional0"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
charge (InputLayer)             [(None, 10, 10, 2)]  0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 8, 8, 64)     1216        charge[0][0]                     
__________________________________________________________________________________________________
conv2d_1 (Conv2D)               (None, 6, 6, 34)     19618       conv2d[0][0]                     
__________________________________________________________________________________________________
conv2d_2 (Conv2D)               (None, 4, 4, 4)      1228        conv2d_1[0][0]                   
______________________________________________________________________________

In [7]:
#Training

#We call fit(), which will train the model by slicing the data into "batches" of size batch_size, 
#and repeatedly iterating over the entire dataset for a given number of epochs.

csv_logger = keras.callbacks.CSVLogger('models/{}'.format(key))
early_stop = EarlyStopping()
callbacks = [early_stop, csv_logger]
callbacks = [csv_logger]
history = model.fit(
    {"charge":x_i[0],"zenith":x_i[1].reshape(-1,1)}, temp_y, epochs=6, validation_split=0.15,callbacks=callbacks)
    

Epoch 1/6
14354/14354 [==============================] - 235s 16ms/step - loss: 0.1239 - mae: 0.2212 - mse: 0.1239 - val_loss: 0.0360 - val_mae: 0.1419 - val_mse: 0.0360
Epoch 2/6
14354/14354 [==============================] - 205s 14ms/step - loss: 0.0357 - mae: 0.1421 - mse: 0.0357 - val_loss: 0.0341 - val_mae: 0.1370 - val_mse: 0.0341
Epoch 3/6
14354/14354 [==============================] - 213s 15ms/step - loss: 0.0316 - mae: 0.1329 - mse: 0.0316 - val_loss: 0.0258 - val_mae: 0.1178 - val_mse: 0.0258
Epoch 4/6
14354/14354 [==============================] - 244s 17ms/step - loss: 0.0298 - mae: 0.1285 - mse: 0.0298 - val_loss: 0.0265 - val_mae: 0.1217 - val_mse: 0.0265
Epoch 5/6
14354/14354 [==============================] - 253s 18ms/step - loss: 0.0286 - mae: 0.1255 - mse: 0.0286 - val_loss: 0.0230 - val_mae: 0.1092 - val_mse: 0.0230
Epoch 6/6
14354/14354 [==============================] - 203s 14ms/step - loss: 0.0279 - mae: 0.1238 - mse: 0.0279 - val_loss: 0.0265 - val_mae: 0.119

In [8]:
#Save an array to a binary file in NumPy .npy format.

np.save("Functional.npy",prep)
model.save('models/model_%s.h5' % key)
#f = open("results.txt", "a")
#now = datetime.now()
#f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    ##now.strftime("%m/%d/%Y %H:%M:%S"),
    #key,
    #len(history.history['loss']),
    #history.history['loss'][len(history.history['loss'])-1],
    #history.history['val_loss'][len(history.history['loss'])-1]
#))
#f.close()